# Actividad integradora — Consigna 1
## Red neuronal convolucional sobre CIFAR-10

**Materia:** Sistemas Inteligentes — Universidad CAECE, sede Mar del Plata

Se elige la red convolucional trabajada en clase (Conv2D 32 y 64, MaxPooling 2x2, densa de 64 y salida de 10).
Sobre esa base se prueban variantes para ver si mejoran las predicciones y qué pasa con la función de pérdida:

| Modelo | Cambio respecto de la base | Motivo |
|---|---|---|
| Base | 32-64-64, densa 64, 15 épocas | Referencia de la consigna |
| A | 32-64-128 con un bloque conv+pool extra | Más profundidad y más filtros: debería captar patrones más complejos |
| B | 16-32, densa 32 | Menos parámetros: sirve para ver si la red sobra o se queda corta |
| C | Base + Dropout(0.3) antes de la densa | Regularización para reducir el sobreajuste |
| D | Base con 30 y 50 épocas | Ver si más entrenamiento sigue mejorando |

Las variantes se justifican en comentarios dentro del código.

In [ ]:
# Importaciones necesarias
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import datasets, layers, models

print("TensorFlow", tf.__version__)
print("GPU disponible:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
# Descarga del set y separación train / test
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

# Normalización: los píxeles vienen de 0 a 255 y se llevan al rango 0-1
train_images = train_images / 255.0
test_images  = test_images / 255.0

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Entrenamiento:", train_images.shape)
print("Prueba:", test_images.shape)
print("Etiquetas de entrenamiento:", train_labels.shape)

# Distribución por clase en el conjunto de entrenamiento
conteo = np.bincount(train_labels.ravel(), minlength=10)
for i, nombre in enumerate(class_names):
    print(f"{nombre:12s} {conteo[i]}")

In [ ]:
# Control visual: primeras 25 imágenes con su clase
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()

In [ ]:
def crear_modelo(filtros=(32, 64, 64), densa=64, dropout=0.0, bloques_extra=0):
    """Arma la red. filtros = cantidad de núcleos de cada capa de convolución.
    bloques_extra agrega pares Conv2D+MaxPooling al final de la parte convolucional."""
    model = models.Sequential()

    # Capa de entrada: 32x32 píxeles y 3 canales (RGB).
    model.add(layers.Conv2D(filtros[0], (3, 3), activation='relu', input_shape=(32, 32, 3)))
    model.add(layers.MaxPooling2D((2, 2)))

    model.add(layers.Conv2D(filtros[1], (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))

    # Tercera convolución: acá el tamaño espacial ya es chico (6x6)
    model.add(layers.Conv2D(filtros[2], (3, 3), activation='relu'))

    for _ in range(bloques_extra):
        model.add(layers.Conv2D(filtros[2], (3, 3), activation='relu', padding='same'))
        model.add(layers.MaxPooling2D((2, 2)))

    # Pasa de 3D a 1D para poder usar capas densas
    model.add(layers.Flatten())
    if dropout > 0:
        model.add(layers.Dropout(dropout))
    model.add(layers.Dense(densa, activation='relu'))
    model.add(layers.Dense(10))
    return model

modelo_base = crear_modelo()
modelo_base.summary()

In [ ]:
# Compilación y entrenamiento de la red base (15 épocas)
modelo_base.compile(optimizer='adam',
                    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                    metrics=['accuracy'])

historial_base = modelo_base.fit(train_images, train_labels, epochs=15,
                                 validation_data=(test_images, test_labels))

In [ ]:
# Curvas de precisión y de pérdida
def graficar(historial, titulo=''):
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(historial.history['accuracy'], label='accuracy')
    plt.plot(historial.history['val_accuracy'], label='val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim([0.5, 1])
    plt.legend(loc='lower right')
    plt.title(titulo)

    plt.subplot(1, 2, 2)
    plt.plot(historial.history['loss'], label='loss')
    plt.plot(historial.history['val_loss'], label='val_loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend(loc='upper right')
    plt.show()

graficar(historial_base, 'Modelo base (15 épocas)')

test_loss, test_acc = modelo_base.evaluate(test_images, test_labels, verbose=2)
print(f"Pérdida en prueba: {test_loss:.4f} — Precisión en prueba: {test_acc:.4f}")

In [ ]:
# Predicciones sobre el conjunto de prueba
predicciones = modelo_base.predict(test_images[:15])

plt.figure(figsize=(12, 6))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(test_images[i])
    clase_predicha = class_names[np.argmax(predicciones[i])]
    clase_real = class_names[test_labels[i][0]]
    color = 'green' if clase_predicha == clase_real else 'red'
    plt.xlabel(f"pred: {clase_predicha}\nreal: {clase_real}", color=color, fontsize=8)
plt.show()

# Matriz de confusión simple para ver qué clases se confunden
pred_todas = np.argmax(modelo_base.predict(test_images), axis=1)
mc = np.zeros((10, 10), dtype=int)
np.add.at(mc, (test_labels.ravel(), pred_todas), 1)
print(mc)

# Las clases con más aciertos y con más confusiones
aciertos = mc.diagonal() / mc.sum(axis=1)
for i, nombre in enumerate(class_names):
    print(f"{nombre:12s} aciertos: {aciertos[i]*100:.1f}%")

### Variante A: más filtros y un bloque convolucional extra
La idea es darle más capacidad a la parte que extrae características. Si el error de entrenamiento baja
pero el de prueba no, la red está memorizando (sobreajuste).

In [ ]:
modelo_a = crear_modelo(filtros=(32, 64, 128), bloques_extra=1)
modelo_a.compile(optimizer='adam',
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
modelo_a.summary()

historial_a = modelo_a.fit(train_images, train_labels, epochs=15,
                           validation_data=(test_images, test_labels))
graficar(historial_a, 'Variante A: 32-64-128 + bloque extra')
loss_a, acc_a = modelo_a.evaluate(test_images, test_labels, verbose=2)
print(f"Variante A — pérdida: {loss_a:.4f} — precisión: {acc_a:.4f}")

### Variante B: menos filtros y menos neuronas
Para comparar hacia el otro lado: si con menos parámetros se obtiene casi la misma precisión,
la red grande no aporta nada.

In [ ]:
modelo_b = crear_modelo(filtros=(16, 32, 32), densa=32)
modelo_b.compile(optimizer='adam',
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
historial_b = modelo_b.fit(train_images, train_labels, epochs=15,
                           validation_data=(test_images, test_labels))
graficar(historial_b, 'Variante B: 16-32-32, densa 32')
loss_b, acc_b = modelo_b.evaluate(test_images, test_labels, verbose=2)
print(f"Variante B — pérdida: {loss_b:.4f} — precisión: {acc_b:.4f}")

### Variante C: dropout antes de la capa densa
El dropout apaga neuronas al azar en cada paso de entrenamiento. Se usa cuando la brecha entre
accuracy y val_accuracy indica sobreajuste.

In [ ]:
modelo_c = crear_modelo(dropout=0.3)
modelo_c.compile(optimizer='adam',
                 loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
historial_c = modelo_c.fit(train_images, train_labels, epochs=15,
                           validation_data=(test_images, test_labels))
graficar(historial_c, 'Variante C: base + Dropout 0.3')
loss_c, acc_c = modelo_c.evaluate(test_images, test_labels, verbose=2)
print(f"Variante C — pérdida: {loss_c:.4f} — precisión: {acc_c:.4f}")

### Variante D: 30 y 50 épocas
Se repite la arquitectura base cambiando solo la cantidad de entrenamientos, para ver si más
épocas siguen bajando la pérdida o si la red ya se estancó.

In [ ]:
resultados_epocas = {}
for ep in (30, 50):
    modelo = crear_modelo()
    modelo.compile(optimizer='adam',
                   loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                   metrics=['accuracy'])
    h = modelo.fit(train_images, train_labels, epochs=ep,
                   validation_data=(test_images, test_labels))
    loss_e, acc_e = modelo.evaluate(test_images, test_labels, verbose=0)
    resultados_epocas[ep] = (h, loss_e, acc_e)
    print(f"{ep} épocas — pérdida: {loss_e:.4f} — precisión: {acc_e:.4f}")
    graficar(h, f'Modelo base con {ep} épocas')

In [ ]:
# Comparación final de todos los modelos
print(f"{'Modelo':<28}{'Pérdida':>10}{'Precisión':>12}")
print("-" * 50)
filas = [('Base (15 épocas)', test_loss, test_acc),
         ('A: 32-64-128 + bloque', loss_a, acc_a),
         ('B: 16-32-32', loss_b, acc_b),
         ('C: base + dropout 0.3', loss_c, acc_c)]
for ep, (h, l, a) in resultados_epocas.items():
    filas.append((f'Base ({ep} épocas)', l, a))
for nombre, l, a in filas:
    print(f"{nombre:<28}{l:>10.4f}{a:>12.4f}")

# Curvas de pérdida de validación superpuestas
plt.figure(figsize=(8, 4))
plt.plot(historial_base.history['val_loss'], label='base')
plt.plot(historial_a.history['val_loss'], label='A')
plt.plot(historial_b.history['val_loss'], label='B')
plt.plot(historial_c.history['val_loss'], label='C')
plt.xlabel('Epoch')
plt.ylabel('Pérdida de validación')
plt.legend()
plt.show()

### Conclusión de la consigna 1
La red base aprende rápido en las primeras épocas y después la mejora se hace muy lenta: la precision
de entrenamiento sigue subiendo mientras la de validación se estanca, que es el síntoma típico de
sobreajuste. Agregar filtros (variante A) no cambia demasiado el resultado y cuesta más tiempo de
entrenamiento, mientras que reducir la red (variante B) empeora un poco la precisión. El dropout
(variante C) reduce la brecha entre entrenamiento y prueba. Pasar de 15 a 30 o 50 épocas mejora poco
y empieza a sobreajustar, así que no conviene seguir entrenando a ciegas.